<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/NLP_Assessment_3_Filtering_and_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateBERT Filtering & Scoring
**Pipeline:** combined.csv → climate-relevance filter → sentiment scoring → outputs

Pulls `combined.csv` directly from GitHub — no Drive mounting, no reprocessing.

**Models used:**
- `climatebert/distilroberta-base-climate-detector` — filters out non-climate text
- `climatebert/distilroberta-base-climate-sentiment` — scores remaining text positive/negative/neutral

**Outputs pushed to `data/processed/` on GitHub:**
- `filtered.csv` — climate-relevant rows only
- `individual_scores.csv` — per-row sentiment scores
- `company_summary.csv` — average sentiment per company (greenwashing signal)

**Before running:**
1. Click the "KEY" icon in the left sidebar
2. Add a secret called `GITHUB_TOKEN` with your personal access token
   - Get one at: github.com/settings/tokens → Generate new token (classic) → tick `repo`
3. Toggle **Notebook access** on

In [ ]:
# --- SETUP ---
!pip install transformers torch sentence-transformers -q

import os
import pandas as pd
import torch
from google.colab import userdata
from transformers import AutoTokenizer

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

OUTPUT_DIR = '/content/outputs'
REPO       = 'Swas26/NLP-ESG'
REPO_DIR   = '/content/repo'
GIT_EMAIL  = 'swas.2625@gmial.com'
GIT_NAME   = 'Swas26'

os.makedirs(OUTPUT_DIR, exist_ok=True)

!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR} -q 2>/dev/null || echo "Repo already cloned"

combined_path = f'{REPO_DIR}/data/processed/combined.csv'
if not os.path.exists(combined_path):
    raise FileNotFoundError('combined.csv not found — run Data_Preprocessing notebook first')

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
print('Setup complete')

In [ ]:
# --- LOAD DATA ---

df = pd.read_csv(combined_path)
print(f'Loaded {len(df)} rows from GitHub')
print(df['source_type'].value_counts())
print('\nPer company:')
print(df.groupby(['company','source_type']).size().unstack(fill_value=0))

In [ ]:
# --- PART 1: CLIMATE RELEVANCE FILTERING ---
# ClimateBERT detector filters out non-climate text
# Keeps only rows where label == 'yes'

from transformers import pipeline

detector = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-detector',
    device=0 if torch.cuda.is_available() else -1
)

tokenizer = AutoTokenizer.from_pretrained('climatebert/distilroberta-base-climate-detector')

def is_climate_related(text):
    try:
        result = detector(str(text), truncation=True, max_length=512)[0]
        return result['label'] == 'yes'
    except Exception as e:
        return False


print('Filtering with ClimateBERT detector...')
df['is_relevant'] = df['text'].apply(is_climate_related)

before = len(df)
filtered_df = df[df['is_relevant'] == True].drop(columns=['is_relevant']).reset_index(drop=True)
after = len(filtered_df)

print(f'Kept {after} / {before} rows ({round(after/before*100)}% climate-relevant)')
print(filtered_df['source_type'].value_counts())
print('\nPer company:')
print(filtered_df.groupby(['company','source_type']).size().unstack(fill_value=0))

filtered_df.to_csv(f'{OUTPUT_DIR}/filtered.csv', index=False)

In [ ]:
# --- PART 2: SENTIMENT SCORING ---

torch.cuda.empty_cache()

sentiment = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-sentiment',
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

def get_sentiment_score(text):
    try:
        results = sentiment(str(text), truncation=True, max_length=512)[0]
        # normalise labels to lowercase to handle 'Positive'/'positive' inconsistency
        score_dict = {item['label'].lower(): item['score'] for item in results}
        p_opp  = score_dict.get('opportunity', 0)
        p_risk = score_dict.get('risk', 0)
        return round(p_opp - p_risk, 4)
    except Exception as e:
        print(f'Error: {e}')
        return None

# quick sanity check on one row before running everything
test = sentiment("We reduced our carbon emissions by 40% in 2024")[0]
print('Label check:', test)

print('\nRunning ClimateBERT sentiment scoring...')
filtered_df['sentiment_score'] = filtered_df['text'].apply(get_sentiment_score)

scored = filtered_df['sentiment_score'].notna().sum()
print(f'Scored {scored} / {len(filtered_df)} rows')
print(filtered_df['sentiment_score'].describe().round(4))

filtered_df.to_csv(f'{OUTPUT_DIR}/individual_scores.csv', index=False)

In [ ]:
results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

print(results.groupby(['company', 'source_type'])['sentiment_score'].mean().round(4).unstack())

In [ ]:
# --- PART 3: CHEAP TALK INDEX ---
# Following Bingler et al. (2023) "How cheap talk in climate disclosures
# relates to climate initiatives, corporate emissions, and reputation risk"
#
# High positive sentiment + high vagueness = strong greenwashing signal
# A company scoring +0.8 sentiment but using mostly vague language
# is more suspicious than one scoring +0.6 with specific, measurable claims.
#
# specificity_score range: -1 (all vague) to +1 (all specific)
# cheap_talk_risk = sentiment_score - specificity_score
#   high value = positive framing built on vague language = greenwashing signal

# vague commitment markers — sounds good, commits to nothing measurable
VAGUE_MARKERS = [
    'committed to', 'aim to', 'working towards', 'we believe',
    'intend to', 'strive to', 'aspire', 'our journey', 'we recognise',
    'dedicated to', 'our goal is', 'we hope', 'in line with',
    'we support', 'we acknowledge', 'sustainability is core',
    'we are focused on', 'we continue to', 'we remain committed',
    'we are proud', 'we look forward', 'we are working',
]

# specific verifiable markers — numbers, past tense achievements, certifications
SPECIFIC_MARKERS = [
    'reduced by', 'increased by', 'achieved', 'certified',
    'verified by', 'third party', 'independently audited',
    'scope 1', 'scope 2', 'scope 3', 'tonnes', 'megawatts',
    'invested $', 'per cent', 'compared to', 'baseline',
    'target of', 'by 2030', 'by 2035', 'by 2040', 'by 2050',
    'fy2023', 'fy2024', 'fy2025', 'in 2023', 'in 2024',
    '%', 'kwh', 'gwh', 'co2e', 'mtco2'
]

def specificity_score(text):
    text_lower = str(text).lower()
    vague_hits    = sum(1 for v in VAGUE_MARKERS    if v in text_lower)
    specific_hits = sum(1 for s in SPECIFIC_MARKERS if s in text_lower)
    total = vague_hits + specific_hits
    if total == 0:
        return 0.0  # neutral — no ESG claim language detected
    return round((specific_hits - vague_hits) / total, 4)

results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

results['specificity_score'] = results['text'].apply(specificity_score)

# cheap talk risk: high sentiment + low specificity = suspicious
# only meaningful for ESG reports (companies writing about themselves)
results['cheap_talk_risk'] = (
    results['sentiment_score'].fillna(0) - results['specificity_score']
).round(4)

print('Specificity score distribution (ESG reports only):')
print(results[results['source_type']=='esg_report']['specificity_score'].describe().round(4))

results.to_csv(f'{OUTPUT_DIR}/individual_scores.csv', index=False)


In [ ]:
# --- PART 4: DIVERGENCE SCORE ---
# FRAMING:
#   ESG reports = self-perception (how the company sees itself)
#   News        = external perception (how the world sees the company)
#
# greenwashing_index = avg_report_sentiment - avg_news_sentiment
#   High positive value = company portrays itself much better than news does
#   Near zero           = consistent narrative across both sources
#


results = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')

report_stats = (
    results[results['source_type'] == 'esg_report']
    .groupby('company')
    .agg(
        report_sentiment=('sentiment_score', 'mean'),
        report_std=('sentiment_score', 'std'),
        report_chunks=('sentiment_score', 'count'),
        avg_cheap_talk_risk=('cheap_talk_risk', 'mean'),
    )
    .round(4)
)

news_stats = (
    results[results['source_type'] == 'news']
    .groupby('company')
    .agg(
        news_sentiment=('sentiment_score', 'mean'),
        news_std=('sentiment_score', 'std'),
        news_chunks=('sentiment_score', 'count'),
    )
    .round(4)
)

divergence = report_stats.join(news_stats)

divergence['adjusted_report_sentiment'] = (
    divergence['report_sentiment'] - divergence['avg_cheap_talk_risk']
).round(4)

divergence['greenwashing_index'] = (
    divergence['adjusted_report_sentiment'] - divergence['news_sentiment']
).round(4)

MIN_REPORT_CHUNKS = 20
MIN_NEWS_CHUNKS   = 5
divergence['data_quality'] = 'ok'
divergence.loc[divergence['report_chunks'] < MIN_REPORT_CHUNKS, 'data_quality'] = 'insufficient_report_data'
divergence.loc[divergence['news_chunks']   < MIN_NEWS_CHUNKS,   'data_quality'] = 'insufficient_news_data'
divergence.loc[divergence['news_sentiment'].isna(), 'data_quality'] = 'insufficient_news_data'

divergence = divergence.sort_values('greenwashing_index', ascending=False)

print('=== GREENWASHING INDEX ((report_sentiment − cheap_talk_risk) − news_sentiment) ===')
print(divergence[['report_sentiment', 'avg_cheap_talk_risk', 'adjusted_report_sentiment',
                   'news_sentiment', 'greenwashing_index',
                   'report_chunks', 'news_chunks', 'data_quality']].to_string())

divergence.to_csv(f'{OUTPUT_DIR}/divergence_scores.csv')
print('\nDivergence scores saved ')


In [ ]:
# --- PART 5: COMPANY SUMMARY ---

results    = pd.read_csv(f'{OUTPUT_DIR}/individual_scores.csv')
divergence = pd.read_csv(f'{OUTPUT_DIR}/divergence_scores.csv', index_col='company')

# ── overall sentiment (all sources) -──────────────────────────────────────────
overall = (
    results
    .groupby('company')['sentiment_score']
    .agg(avg_sentiment='mean', total_chunks='count')
    .round(4)
)

# ── specificity from ESG reports  -----────────────────────────────────────────
specificity = (
    results[results['source_type'] == 'esg_report']
    .groupby('company')
    .agg(avg_specificity=('specificity_score', 'mean'))
    .round(4)
)

# ── merge all signals ─────────────────────────────────────────────────────────
summary = (
    overall
    .join(specificity)
    .join(divergence[[
        'report_sentiment', 'news_sentiment',
        'greenwashing_index', 'report_chunks',
        'news_chunks', 'data_quality'
    ]])
    .sort_values('greenwashing_index', ascending=False)
)

print('=== FINAL COMPANY SUMMARY ===\n')
print(summary.to_string())

summary.to_csv(f'{OUTPUT_DIR}/company_summary.csv')
print('\nFull summary saved ')

print('''
--- INTERPRETATION ---
greenwashing_index  (report_sentiment − cheap_talk_risk) − news_sentiment
                    HIGH  → company over-claims vs news even after checking vague language
                    ~0    → narrative is consistent across ESG report and news
avg_specificity     HIGH  → language is specific and verifiable (good)
                    LOW   → language is vague and aspirational (suspicious)
data_quality        ok                   → sufficient data, result is reliable
                    insufficient_*_data  → too few chunks, treat with caution
''')

In [ ]:
# --- FINDINGS SUMMARY FOR PRESENTATION ---

reliable = summary[
    (summary['data_quality'] == 'ok') &
    (summary['news_chunks'] >= 5) &
    (summary['greenwashing_index'].notna())
].copy()

print('=== RELIABLE RESULTS (sufficient data, both sources) ===\n')
print(reliable[['report_sentiment', 'news_sentiment',
                 'greenwashing_index']].to_string())

print('\n=== RANKING BY GREENWASHING RISK ===')
for company, row in reliable.iterrows():
    gi = row['greenwashing_index']
    if gi > 0.3:
        flag = 'HIGH RISK'
    elif gi > 0.1:
        flag = 'MODERATE'
    else:
        flag = 'LOW RISK'
    print(f"{flag}  {company:<25} GI={gi:+.3f}  specificity={sp:+.3f}")

In [ ]:
# --- Greenwashing Indicator ---

import numpy as np

THRESHOLD = 0.6
# THRESHOLD = reliable['gw_probability'].median()

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

reliable = summary[summary['data_quality'] == 'ok'].copy()
reliable['gw_probability'] = reliable['greenwashing_index'].apply(sigmoid).round(4)
reliable['verdict'] = reliable['gw_probability'].apply(
    lambda p: 'Greenwashing' if p >= THRESHOLD else 'Not-Greenwashing'
)

print('=== GREENWASHING Indicator (reliable companies only) ===\n')
print(f'threshold: {THRESHOLD}\n')
print(reliable[['greenwashing_index', 'verdict']].to_string())

In [ ]:
# --- PUSH OUTPUTS TO GITHUB ---
import shutil

processed_dir = f'{REPO_DIR}/data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.csv'):
        shutil.copy(f'{OUTPUT_DIR}/{f}', f'{processed_dir}/{f}')
        print(f'Copied {f}')

!cd {REPO_DIR} && git config user.email "{GIT_EMAIL}"
!cd {REPO_DIR} && git config user.name "{GIT_NAME}"
!cd {REPO_DIR} && git add data/processed/
!cd {REPO_DIR} && git commit -m "update scoring outputs with divergence and cheap talk scores"
!cd {REPO_DIR} && git push

print('\nAll outputs pushed to GitHub → data/processed/')